# Spectral dispersion traces — Euclid-like

This notebook explores spectral dispersion physics at Euclid NISP scale.
It builds realistic scenes with mixed source types and uses the `ForwardModel`
with multi-order dispersion (`[0, 1, 2]`) to simulate K-sequence spectrograms.

Topics covered
--------------
1. K-sequence overview: the four dispersion directions and GWA ±4° tilt.
2. Dispersion trace computation: mapping λ → (x_det, y_det) for each K-step.
3. Multi-order contamination: 0th-order peanut + 1st + 2nd order overlap.
4. Source confusion: blended vs isolated sources in spectrograms.
5. Forward model pixel budget: understanding how padded detector images
   relate to the scene cube for the ML models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spectangle.physics.dispersion import KSequence
from spectangle.physics.psf import PSFModel
from spectangle.simulations.forward import ForwardModel
from spectangle.simulations.sed import BlackbodySED

# Euclid-like miniature setup (128x128 scene, 128 spectral channels)
NY, NX = 128, 128
N_LAMBDA = 128
wav = np.linspace(9250, 18500, N_LAMBDA)

kseq = KSequence.miniature(N_LAMBDA)
psf  = PSFModel(fwhm_pixels=1.6)

print('K-sequence dispersion vectors:')
labels = ['RGS000+0', 'RGS180+4', 'RGS000-4', 'RGS180+0']
for i, (d, lbl) in enumerate(zip(kseq, labels)):
    ref_off = d.wavelength_to_offset(wav[0])
    max_off = d.wavelength_to_offset(wav[-1])
    print(f'  K{i+1} ({lbl}): dx_max={max_off.delta_x:.1f}px  dy_max={max_off.delta_y:.1f}px')

## 1. Visualise the four K-sequence dispersion directions
Plot the dispersion trace on a 2-D canvas for a point source at the field
centre.  The four arms diverge in slightly different directions: two are
nearly horizontal (RGS000/180) and two have a ±4° tilt from the GWA.

In [ ]:
fwd = ForwardModel(ksequence=kseq, psf_model=psf, image_shape=(NY, NX), orders=[1])
pad_y, pad_x = fwd.pad_y, fwd.pad_x
H_spec, W_spec = fwd.spectrogram_shape

# Source at centre of scene
src_x, src_y = NX // 2, NY // 2

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cmap_bg = plt.cm.Greys
for k, (disp, lbl) in enumerate(zip(kseq, labels)):
    ax = axes[k]
    # Background: blank canvas
    canvas = np.zeros((H_spec, W_spec))
    im = ax.imshow(canvas, origin='lower', cmap='inferno', vmin=0, vmax=1)

    # Compute trace coordinates
    offsets = np.array([disp.wavelength_to_offset(l) for l in wav])
    trace_x = src_x + pad_x + offsets[:, 0]
    trace_y = src_y + pad_y + offsets[:, 1]

    # Colour by wavelength
    sc = ax.scatter(trace_x, trace_y, c=wav, cmap='rainbow', s=3, zorder=5)
    ax.scatter([src_x + pad_x], [src_y + pad_y], c='white', s=50, zorder=10, label='source')
    ax.set_title(lbl, fontsize=10)
    ax.set_xlim(0, W_spec); ax.set_ylim(0, H_spec)
    ax.set_xlabel('x [pix]'); ax.set_ylabel('y [pix]')
    plt.colorbar(sc, ax=ax, label='λ (Å)', fraction=0.04)

plt.suptitle('K-sequence dispersion traces (1st order, centre source)', fontsize=12)
plt.tight_layout(); plt.show()

print(f'\nScene: {NY}x{NX} pixels')
print(f'Padding: pad_y={pad_y}, pad_x={pad_x}')
print(f'Spectrogram shape: {H_spec}x{W_spec} pixels')

## 2. Multi-order contamination
Simulate 0th, 1st and 2nd orders for a bright point source and show how
each order contributes to the spectrogram.

In [ ]:
# Build a single bright point source cube (blackbody at 7000 K)
cube = np.zeros((N_LAMBDA, NY, NX), dtype=np.float32)
sed = BlackbodySED(7000)
flux = sed(wav).astype(np.float32)
cube[:, NY//2, NX//2] = flux * 10.0
cube = psf.convolve_cube(cube)

# Simulate orders separately
results = {}
for orders in [[0], [1], [2], [0, 1, 2]]:
    fwd_ord = ForwardModel(ksequence=kseq, psf_model=psf, image_shape=(NY, NX), orders=orders)
    results[str(orders)] = fwd_ord(cube, wav)

# Show first K-step for each order combination
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
titles = ['0th order', '1st order', '2nd order', 'All orders (0,1,2)']
for ax, (key, title) in zip(axes, zip(results.keys(), titles)):
    img = np.log1p(results[key][0])
    im = ax.imshow(img, origin='lower', cmap='inferno')
    ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Diffraction orders for K-step 1 (RGS000+0)', fontsize=12)
plt.tight_layout(); plt.show()

## 3. Source confusion in crowded fields
Demonstrate spectral overlap between two nearby sources with different
effective temperatures.  Their traces cross in some K-steps but are
separated in others — this complementary information is what the
network exploits to disentangle blended spectra.

In [ ]:
# Two nearby sources
cube2 = np.zeros((N_LAMBDA, NY, NX), dtype=np.float32)

# Hot star (15000 K) at (70, 64)
hot = BlackbodySED(15000)
cube2[:, 64, 70] = hot(wav) * 5.0

# Cool star (3800 K) at (58, 64) — 12 pixels apart
cool = BlackbodySED(3800)
cube2[:, 64, 58] = cool(wav) * 8.0

cube2 = psf.convolve_cube(cube2)

fwd1 = ForwardModel(ksequence=kseq, psf_model=psf, image_shape=(NY, NX), orders=[1])
specs2 = fwd1(cube2, wav)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for k in range(4):
    ax = axes[k]
    im = ax.imshow(np.log1p(specs2[k]), origin='lower', cmap='inferno')
    ax.set_title(labels[k]); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Two nearby sources (ΔT=11200K, Δx=12px) — log-stretch', fontsize=12)
plt.tight_layout(); plt.show()

# Highlight spectral difference using colour coding
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wav, hot(wav)  / hot(wav).max(),  label='Hot (15000K)',  color='firebrick')
ax.plot(wav, cool(wav) / cool(wav).max(), label='Cool (3800K)',  color='steelblue')
ax.set_xlabel('Wavelength (Å)'); ax.set_ylabel('Normalised flux')
ax.set_title('SEDs of the two confused sources'); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

## 4. Pixel budget and padding analysis
Summarise how the spectrogram size scales with scene size, spectral
sampling and dispersion parameters.

In [ ]:
print('Forward model geometry summary')
print('=' * 50)
print(f'  Scene size        : {NY} x {NX} px')
print(f'  n_lambda          : {N_LAMBDA}')
print(f'  Wavelength range  : {wav[0]:.0f} – {wav[-1]:.0f} Å')
print(f'  PSF FWHM          : {psf.fwhm_pixels} px')
print(f'  Orders simulated  : [0, 1, 2]')
print()

fwd_multi = ForwardModel(ksequence=kseq, psf_model=psf, image_shape=(NY, NX), orders=[0,1,2])
print(f'  pad_x (multi-order): {fwd_multi.pad_x} px  (each side)')
print(f'  pad_y (multi-order): {fwd_multi.pad_y} px  (each side)')
print(f'  Spectrogram shape  : {fwd_multi.spectrogram_shape[0]} x {fwd_multi.spectrogram_shape[1]} px')
print()

# Compare scene vs spectrogram pixel count
n_scene = NY * NX * N_LAMBDA
n_spec  = 4 * fwd_multi.spectrogram_shape[0] * fwd_multi.spectrogram_shape[1]
ratio   = n_spec / n_scene
print(f'  Scene cube voxels  : {n_scene:,}  (ny x nx x n_lambda)')
print(f'  Spectrogram pixels : {n_spec:,}  (4 x H_spec x W_spec)')
print(f'  Input/output ratio : {ratio:.2f}x  (underdetermined system!)')
print()
print('  Note: The reconstruction is an ill-posed inverse problem because')
print('  the 4 spectrograms contain less information than the 3D cube.')
print('  The ML models implicitly learn a regularised solution; the PINN')
print('  physics loss explicitly enforces the forward-model constraint.')

## 5. Loading a simulated complex dataset and checking traces
Load the HDF5 generated in notebook `01_generate_euclid_like_dataset.ipynb`
and overlay computed traces on the actual spectrograms.

In [ ]:
from pathlib import Path
from spectangle.simulations.io import load_simulation

h5_path = Path('data/raw/complex_euclid_50s.h5')
if not h5_path.exists():
    print(f'File not found: {h5_path}')
    print('Run 01_generate_euclid_like_dataset.ipynb first.')
else:
    data = load_simulation(h5_path, indices=[0])
    s0    = data['samples'][0]
    specs = s0['spectrograms']   # (4, H_spec, W_spec)
    cube0 = s0['cube']           # (n_lambda, ny, nx)
    wav_h5 = data['wavelengths']

    # Load metadata for geometry
    meta = data['metadata']
    pad_x_h5 = int(float(meta.get('pad_x', pad_x)))
    pad_y_h5 = int(float(meta.get('pad_y', pad_y)))

    # Find brightest source position from broadband cube
    broad = cube0.sum(0)
    iy, ix = np.unravel_index(np.argmax(broad), broad.shape)
    print(f'Brightest source at scene coords: (x={ix}, y={iy})')

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for k, (disp, lbl) in enumerate(zip(kseq, labels)):
        ax = axes[k]
        ax.imshow(np.log1p(specs[k]), origin='lower', cmap='inferno')
        ax.set_title(lbl); ax.axis('off')

        # Overlay trace for brightest source
        offsets_k = np.array([disp.wavelength_to_offset(l) for l in wav_h5])
        tx = ix + pad_x_h5 + offsets_k[:, 0]
        ty = iy + pad_y_h5 + offsets_k[:, 1]
        ax.plot(tx, ty, color='cyan', lw=1.0, alpha=0.8)

    plt.suptitle('Complex dataset: spectrograms with brightest-source trace overlay')
    plt.tight_layout(); plt.show()

## Key takeaways
- The four K-sequence dispersion directions are complementary: sources blended
  in one K-step may be separated in another, giving the network enough information
  to disentangle them.
- Multi-order contamination (0th peanut, 2nd order) adds realism and difficulty;
  the ComplexSimulator includes these automatically.
- The reconstruction problem is genuinely ill-posed (more unknowns than
  observations), making physical priors (PINN) particularly important.

Next: train models on the euclid-like dataset (notebooks in `training_testing/`).